In [256]:
import random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, accuracy_score

def set_seed(seed=97798760):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # make cudnn deterministic
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(507)

# T1
t1_train = pd.read_csv("../multi_omics/T1_train_selected.csv")
t1_test = pd.read_csv("../multi_omics/T1_test_selected.csv")

# T2
t2_train = pd.read_csv("../multi_omics/T2_train_selected.csv")
t2_test = pd.read_csv("../multi_omics/T2_test_selected.csv")

# ADC
adc_train = pd.read_csv("../multi_omics/ADC_train_selected.csv")
adc_test = pd.read_csv("../multi_omics/ADC_test_selected.csv")

def split_X_y(df):
    X = df.drop(columns=["label"])
    y = df["label"]
    return X, y

X_t1_train, y_t1_train = split_X_y(t1_train)
X_t1_test, y_t1_test = split_X_y(t1_test)

X_t2_train, y_t2_train = split_X_y(t2_train)
X_t2_test, y_t2_test = split_X_y(t2_test)

X_adc_train, y_adc_train = split_X_y(adc_train)
X_adc_test, y_adc_test = split_X_y(adc_test)

class MultiModalDataset(Dataset):
    def __init__(self, X_t1, X_t2, X_adc, y):
        self.X_t1 = torch.tensor(X_t1.values, dtype=torch.float32)
        self.X_t2 = torch.tensor(X_t2.values, dtype=torch.float32)
        self.X_adc = torch.tensor(X_adc.values, dtype=torch.float32)
        self.y = torch.tensor(y.values, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            self.X_t1[idx],
            self.X_t2[idx],
            self.X_adc[idx],
            self.y[idx]
        )


train_dataset = MultiModalDataset(
    X_t1_train, X_t2_train, X_adc_train, y_t1_train
)

test_dataset = MultiModalDataset(
    X_t1_test, X_t2_test, X_adc_test, y_t1_test
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)

# Single Modality Neural Network

In [257]:
class SingleModalityNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=8, clf_hidden_dim=8, dropout=0.4):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, clf_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(clf_hidden_dim, 1)
        )

    def forward(self, x):
        h = self.encoder(x)
        logit = self.classifier(h).squeeze(1)
        return logit

def eval_single_from_multimodal(model, loader, device, modality="t1"):
    model.eval()

    all_y, all_prob, all_pred = [], [], []

    with torch.no_grad():
        for x_t1, x_t2, x_adc, y in loader:
            x_t1, x_t2, x_adc, y = (
                x_t1.to(device),
                x_t2.to(device),
                x_adc.to(device),
                y.to(device)
            )

            if modality == "t1":
                x = x_t1
            elif modality == "t2":
                x = x_t2
            elif modality == "adc":
                x = x_adc

            logits = model(x)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            all_y.extend(y.cpu().numpy())
            all_prob.extend(probs.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())

    all_y = np.array(all_y)
    all_prob = np.array(all_prob)
    all_pred = np.array(all_pred)

    acc = accuracy_score(all_y, all_pred)
    auc = roc_auc_score(all_y, all_prob) if len(np.unique(all_y)) > 1 else np.nan

    return acc, auc

def train_single_from_multimodal(
    model,
    train_loader,
    test_loader,
    device,
    modality="t1",
    epochs=20,
    lr=1e-3,
    weight_decay=1e-3
):
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x_t1, x_t2, x_adc, y in train_loader:
            x_t1, x_t2, x_adc, y = (
                x_t1.to(device),
                x_t2.to(device),
                x_adc.to(device),
                y.to(device)
            )

            # pick modality
            if modality == "t1":
                x = x_t1
            elif modality == "t2":
                x = x_t2
            elif modality == "adc":
                x = x_adc
            else:
                raise ValueError("modality must be 't1', 't2', or 'adc'")

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            total_loss += loss.item() * y.size(0)

        total_loss /= len(train_loader.dataset)

        train_acc, train_auc = eval_single_from_multimodal(model, train_loader, device, modality)
        test_acc, test_auc = eval_single_from_multimodal(model, test_loader, device, modality)

        if epoch == 0 or (epoch+1) % 10 == 0 :
            print(
                f"Epoch {epoch+1:03d} | Loss {total_loss:.4f} | "
                f"Train AUC {train_auc:.4f} | Train ACC {train_acc:.4f} |"
                f"Test AUC {test_auc:.4f} | Test ACC {test_acc:.4f}"
            )

    return (train_acc, train_auc, test_acc, test_auc)

In [258]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

t1_model = SingleModalityNet(X_t1_train.shape[1])

train_single_from_multimodal(
    t1_model,
    train_loader,
    test_loader,
    device,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=200,
    modality="t1"
)

Epoch 001 | Loss 0.6506 | Train AUC 0.7912 | Train ACC 0.7947 |Test AUC 0.7163 | Test ACC 0.8333
Epoch 010 | Loss 0.4925 | Train AUC 0.8052 | Train ACC 0.8973 |Test AUC 0.8523 | Test ACC 0.8788
Epoch 020 | Loss 0.3851 | Train AUC 0.8450 | Train ACC 0.8935 |Test AUC 0.8732 | Test ACC 0.8788
Epoch 030 | Loss 0.3639 | Train AUC 0.8744 | Train ACC 0.8935 |Test AUC 0.8797 | Test ACC 0.8788
Epoch 040 | Loss 0.3335 | Train AUC 0.8848 | Train ACC 0.8935 |Test AUC 0.8928 | Test ACC 0.8788
Epoch 050 | Loss 0.3369 | Train AUC 0.8928 | Train ACC 0.8935 |Test AUC 0.8915 | Test ACC 0.8788
Epoch 060 | Loss 0.3148 | Train AUC 0.8940 | Train ACC 0.8935 |Test AUC 0.8954 | Test ACC 0.8788
Epoch 070 | Loss 0.2993 | Train AUC 0.8958 | Train ACC 0.8973 |Test AUC 0.8902 | Test ACC 0.8788
Epoch 080 | Loss 0.3166 | Train AUC 0.8990 | Train ACC 0.8973 |Test AUC 0.8784 | Test ACC 0.8788
Epoch 090 | Loss 0.2833 | Train AUC 0.8995 | Train ACC 0.8973 |Test AUC 0.8745 | Test ACC 0.8788
Epoch 100 | Loss 0.2960 | Trai

(0.8935361216730038,
 0.9048708072540522,
 0.8787878787878788,
 0.8718954248366013)

In [259]:
t2_model = SingleModalityNet(X_t2_train.shape[1])

train_single_from_multimodal(
    t2_model,
    train_loader,
    test_loader,
    device,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=200,
    modality="t2"
)

Epoch 001 | Loss 0.7002 | Train AUC 0.7492 | Train ACC 0.4144 |Test AUC 0.6235 | Test ACC 0.3485
Epoch 010 | Loss 0.5018 | Train AUC 0.8609 | Train ACC 0.8859 |Test AUC 0.8327 | Test ACC 0.8333
Epoch 020 | Loss 0.4423 | Train AUC 0.8523 | Train ACC 0.8859 |Test AUC 0.8248 | Test ACC 0.8333
Epoch 030 | Loss 0.4048 | Train AUC 0.8613 | Train ACC 0.8859 |Test AUC 0.8301 | Test ACC 0.8182
Epoch 040 | Loss 0.3960 | Train AUC 0.8681 | Train ACC 0.8859 |Test AUC 0.8366 | Test ACC 0.8182
Epoch 050 | Loss 0.3955 | Train AUC 0.8732 | Train ACC 0.8859 |Test AUC 0.8392 | Test ACC 0.8182
Epoch 060 | Loss 0.3774 | Train AUC 0.8759 | Train ACC 0.8859 |Test AUC 0.8471 | Test ACC 0.8182
Epoch 070 | Loss 0.3717 | Train AUC 0.8826 | Train ACC 0.8859 |Test AUC 0.8484 | Test ACC 0.8182
Epoch 080 | Loss 0.3519 | Train AUC 0.8909 | Train ACC 0.8859 |Test AUC 0.8444 | Test ACC 0.8182
Epoch 090 | Loss 0.3381 | Train AUC 0.8741 | Train ACC 0.8859 |Test AUC 0.8065 | Test ACC 0.8182
Epoch 100 | Loss 0.3432 | Trai

(0.8897338403041825, 0.851067244423046, 0.8181818181818182, 0.7915032679738561)

In [260]:
adc_model = SingleModalityNet(X_adc_train.shape[1])

train_single_from_multimodal(
    adc_model,
    train_loader,
    test_loader,
    device,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=200,
    modality="adc"
)

Epoch 001 | Loss 0.6971 | Train AUC 0.8256 | Train ACC 0.5475 |Test AUC 0.7320 | Test ACC 0.5909
Epoch 010 | Loss 0.5839 | Train AUC 0.8863 | Train ACC 0.8555 |Test AUC 0.8170 | Test ACC 0.8485
Epoch 020 | Loss 0.4940 | Train AUC 0.8958 | Train ACC 0.8745 |Test AUC 0.8026 | Test ACC 0.8333
Epoch 030 | Loss 0.4296 | Train AUC 0.9046 | Train ACC 0.8745 |Test AUC 0.8039 | Test ACC 0.8333
Epoch 040 | Loss 0.3857 | Train AUC 0.8832 | Train ACC 0.8745 |Test AUC 0.8235 | Test ACC 0.8333
Epoch 050 | Loss 0.4043 | Train AUC 0.8557 | Train ACC 0.8745 |Test AUC 0.7843 | Test ACC 0.8333
Epoch 060 | Loss 0.3689 | Train AUC 0.8926 | Train ACC 0.8745 |Test AUC 0.8203 | Test ACC 0.8333
Epoch 070 | Loss 0.3613 | Train AUC 0.8874 | Train ACC 0.8783 |Test AUC 0.8444 | Test ACC 0.8333
Epoch 080 | Loss 0.3686 | Train AUC 0.8915 | Train ACC 0.8783 |Test AUC 0.8261 | Test ACC 0.8333
Epoch 090 | Loss 0.3668 | Train AUC 0.8887 | Train ACC 0.8783 |Test AUC 0.7908 | Test ACC 0.8333
Epoch 100 | Loss 0.3350 | Trai

(0.8745247148288974,
 0.8898250682073503,
 0.8484848484848485,
 0.8294117647058824)

# Non-Adaptive Early Fusion Network

In [261]:
class ModalityEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=8, dropout=0.4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        return self.net(x)
    
class EncodedConcatFusionNet(nn.Module):
    def __init__(
        self,
        t1_dim,
        t2_dim,
        adc_dim,
        hidden_dim=16,
        clf_hidden_dim=16,
        dropout=0.2
    ):
        super().__init__()

        self.t1_encoder = ModalityEncoder(t1_dim, hidden_dim, dropout)
        self.t2_encoder = ModalityEncoder(t2_dim, hidden_dim, dropout)
        self.adc_encoder = ModalityEncoder(adc_dim, hidden_dim, dropout)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 3, clf_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(clf_hidden_dim, 1)
        )

    def forward(self, x_t1, x_t2, x_adc):
        h_t1 = self.t1_encoder(x_t1)
        h_t2 = self.t2_encoder(x_t2)
        h_adc = self.adc_encoder(x_adc)

        h_fused = torch.cat([h_t1, h_t2, h_adc], dim=1)
        logit = self.classifier(h_fused).squeeze(1)

        return logit

In [262]:
def evaluate_concat_model(model, loader, device):
    model.eval()

    all_y, all_prob, all_pred = [], [], []

    with torch.no_grad():
        for x_t1, x_t2, x_adc, y in loader:
            x_t1 = x_t1.to(device)
            x_t2 = x_t2.to(device)
            x_adc = x_adc.to(device)
            y = y.to(device)

            logits = model(x_t1, x_t2, x_adc)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            all_y.extend(y.cpu().numpy())
            all_prob.extend(probs.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())

    all_y = np.array(all_y)
    all_prob = np.array(all_prob)
    all_pred = np.array(all_pred)

    acc = accuracy_score(all_y, all_pred)
    auc = roc_auc_score(all_y, all_prob) if len(np.unique(all_y)) > 1 else np.nan

    return {
        "acc": acc,
        "auc": auc,
        "y_true": all_y,
        "y_prob": all_prob,
        "y_pred": all_pred
    }

def train_concat_model(model, train_loader, test_loader, device, epochs=100, lr=1e-3, weight_decay=1e-4):
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for x_t1, x_t2, x_adc, y in train_loader:
            x_t1 = x_t1.to(device)
            x_t2 = x_t2.to(device)
            x_adc = x_adc.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(x_t1, x_t2, x_adc)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * y.size(0)

        total_loss /= len(train_loader.dataset)

        train_result = evaluate_concat_model(model, train_loader, device)
        test_result = evaluate_concat_model(model, test_loader, device)

        if epoch == 0 or (epoch+1) % 10 == 0:
            print(
                f"Epoch {epoch+1:03d} | "
                f"Loss {total_loss:.4f} | "
                f"Train AUC {train_result['auc']:.4f} | "
                f"Train ACC {train_result['acc']:.4f} | "
                f"Test AUC {test_result['auc']:.4f} | "
                f"Test ACC {test_result['acc']:.4f}"
            )

    return model


In [263]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
concat_model = EncodedConcatFusionNet(
    t1_dim=X_t1_train.shape[1],
    t2_dim=X_t2_train.shape[1],
    adc_dim=X_adc_train.shape[1],
    hidden_dim=8,
    clf_hidden_dim=8,
    dropout=0.4
)

concat_model = train_concat_model(
    concat_model,
    train_loader,
    test_loader,
    device,
    epochs=200,
    lr=1e-3,
    weight_decay=1e-4
)

Epoch 001 | Loss 0.6832 | Train AUC 0.5739 | Train ACC 0.7681 | Test AUC 0.3621 | Test ACC 0.7727
Epoch 010 | Loss 0.5064 | Train AUC 0.7788 | Train ACC 0.8897 | Test AUC 0.8105 | Test ACC 0.8788
Epoch 020 | Loss 0.3534 | Train AUC 0.8927 | Train ACC 0.8973 | Test AUC 0.9216 | Test ACC 0.8788
Epoch 030 | Loss 0.2998 | Train AUC 0.8987 | Train ACC 0.9011 | Test AUC 0.9124 | Test ACC 0.8788
Epoch 040 | Loss 0.2937 | Train AUC 0.9046 | Train ACC 0.9049 | Test AUC 0.9059 | Test ACC 0.8788
Epoch 050 | Loss 0.3033 | Train AUC 0.9117 | Train ACC 0.9049 | Test AUC 0.8954 | Test ACC 0.8636
Epoch 060 | Loss 0.2683 | Train AUC 0.9146 | Train ACC 0.9049 | Test AUC 0.8967 | Test ACC 0.8636
Epoch 070 | Loss 0.2770 | Train AUC 0.9183 | Train ACC 0.9011 | Test AUC 0.9020 | Test ACC 0.8636
Epoch 080 | Loss 0.2838 | Train AUC 0.9219 | Train ACC 0.9049 | Test AUC 0.8980 | Test ACC 0.8636
Epoch 090 | Loss 0.2994 | Train AUC 0.9253 | Train ACC 0.9011 | Test AUC 0.8928 | Test ACC 0.8636
Epoch 100 | Loss 0.2

# Adaptive Global Weight Fusion Network

In [264]:
class GlobalWeightFusionNet(nn.Module):
    def __init__(
        self,
        t1_dim,
        t2_dim,
        adc_dim,
        hidden_dim=8,
        clf_hidden_dim=8,
        dropout=0.4
    ):
        super().__init__()

        self.t1_encoder = ModalityEncoder(t1_dim, hidden_dim, dropout)
        self.t2_encoder = ModalityEncoder(t2_dim, hidden_dim, dropout)
        self.adc_encoder = ModalityEncoder(adc_dim, hidden_dim, dropout)

        # trainable global fusion logits -> softmax -> global weights
        self.global_gate_logits = nn.Parameter(torch.zeros(3))

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, clf_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(clf_hidden_dim, 1)
        )

    def forward(self, x_t1, x_t2, x_adc):
        h_t1 = self.t1_encoder(x_t1)
        h_t2 = self.t2_encoder(x_t2)
        h_adc = self.adc_encoder(x_adc)

        global_weights = torch.softmax(self.global_gate_logits, dim=0)  # shape: (3,)

        h_fused = (
            global_weights[0] * h_t1 +
            global_weights[1] * h_t2 +
            global_weights[2] * h_adc
        )

        logit = self.classifier(h_fused).squeeze(1)

        return logit, global_weights

In [265]:
def evaluate_global_weight_model(model, loader, device):
    model.eval()

    all_y, all_prob, all_pred = [], [], []
    learned_weights = None

    with torch.no_grad():
        for x_t1, x_t2, x_adc, y in loader:
            x_t1 = x_t1.to(device)
            x_t2 = x_t2.to(device)
            x_adc = x_adc.to(device)
            y = y.to(device)

            logits, global_weights = model(x_t1, x_t2, x_adc)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            all_y.extend(y.cpu().numpy())
            all_prob.extend(probs.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())

            learned_weights = global_weights.detach().cpu().numpy()

    all_y = np.array(all_y)
    all_prob = np.array(all_prob)
    all_pred = np.array(all_pred)

    acc = accuracy_score(all_y, all_pred)
    auc = roc_auc_score(all_y, all_prob) if len(np.unique(all_y)) > 1 else np.nan

    return {
        "acc": acc,
        "auc": auc,
        "global_weights": learned_weights,
        "y_true": all_y,
        "y_prob": all_prob,
        "y_pred": all_pred
    }

def train_global_weight_model(model, train_loader, test_loader, device, epochs=100, lr=1e-3, weight_decay=1e-4):
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for x_t1, x_t2, x_adc, y in train_loader:
            x_t1 = x_t1.to(device)
            x_t2 = x_t2.to(device)
            x_adc = x_adc.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits, global_weights = model(x_t1, x_t2, x_adc)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * y.size(0)

        total_loss /= len(train_loader.dataset)

        train_result = evaluate_global_weight_model(model, train_loader, device)
        test_result = evaluate_global_weight_model(model, test_loader, device)

        if epoch == 0 or (epoch+1) % 1 == 0:
            print(
                f"Epoch {epoch+1:03d} | "
                f"Loss {total_loss:.4f} | "
                f"Train AUC {train_result['auc']:.4f} | "
                f"Train ACC {train_result['acc']:.4f} | "
                # check confusion matrix if needed
                # f"Train Confusion Matrix:\n{pd.crosstab(train_result['y_true'], train_result['y_pred'], rownames=['True'], colnames=['Pred'])}\n"
                f"Test AUC {test_result['auc']:.4f} | "
                f"Test ACC {test_result['acc']:.4f}"
                #f"Test Confusion Matrix:\n{pd.crosstab(test_result['y_true'], test_result['y_pred'], rownames=['True'], colnames=['Pred'])}\n"
            )
            print("Global weights [T1, T2, ADC]:", np.round(test_result["global_weights"], 4))

    return model

In [283]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

global_model = GlobalWeightFusionNet(
    t1_dim=X_t1_train.shape[1],
    t2_dim=X_t2_train.shape[1],
    adc_dim=X_adc_train.shape[1],
    hidden_dim=8,
    clf_hidden_dim=8,
    dropout=0.4
)

global_model = train_global_weight_model(
    global_model,
    train_loader,
    test_loader,
    device,
    epochs=200,
    lr=1e-3,
    weight_decay=1e-4
)

Epoch 001 | Loss 0.7681 | Train AUC 0.7463 | Train ACC 0.2357 | Test AUC 0.6889 | Test ACC 0.2273
Global weights [T1, T2, ADC]: [0.335  0.3333 0.3317]
Epoch 002 | Loss 0.7425 | Train AUC 0.8941 | Train ACC 0.2357 | Test AUC 0.9020 | Test ACC 0.2273
Global weights [T1, T2, ADC]: [0.3371 0.3327 0.3302]
Epoch 003 | Loss 0.7164 | Train AUC 0.8918 | Train ACC 0.4030 | Test AUC 0.9085 | Test ACC 0.3939
Global weights [T1, T2, ADC]: [0.339  0.3312 0.3298]
Epoch 004 | Loss 0.6998 | Train AUC 0.8910 | Train ACC 0.5285 | Test AUC 0.9163 | Test ACC 0.5758
Global weights [T1, T2, ADC]: [0.3405 0.3302 0.3292]
Epoch 005 | Loss 0.6720 | Train AUC 0.8900 | Train ACC 0.5399 | Test AUC 0.9242 | Test ACC 0.6061
Global weights [T1, T2, ADC]: [0.3406 0.3297 0.3297]
Epoch 006 | Loss 0.6399 | Train AUC 0.8877 | Train ACC 0.7452 | Test AUC 0.9294 | Test ACC 0.7727
Global weights [T1, T2, ADC]: [0.3401 0.3293 0.3306]
Epoch 007 | Loss 0.6055 | Train AUC 0.8865 | Train ACC 0.8517 | Test AUC 0.9373 | Test ACC 0.9

# Adaptive Gated Weight Fusion Network

In [267]:
class GatedAdaptiveFusionNet(nn.Module):
    def __init__(
        self,
        t1_dim,
        t2_dim,
        adc_dim,
        hidden_dim=16,
        gate_hidden_dim=8,
        clf_hidden_dim=8,
        dropout=0.4
    ):
        super().__init__()

        # modality-specific encoders
        self.t1_encoder = ModalityEncoder(t1_dim, hidden_dim, dropout)
        self.t2_encoder = ModalityEncoder(t2_dim, hidden_dim, dropout)
        self.adc_encoder = ModalityEncoder(adc_dim, hidden_dim, dropout)

        # gate network: outputs 3 modality weights
        self.gate_net = nn.Sequential(
            nn.Linear(hidden_dim * 3, gate_hidden_dim),
            nn.ReLU(),
            nn.Linear(gate_hidden_dim, 3)
        )

        # classifier on fused representation
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, clf_hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(clf_hidden_dim, 1)
        )

    def forward(self, x_t1, x_t2, x_adc):
        # encode each modality
        h_t1 = self.t1_encoder(x_t1)    # (B, hidden_dim)
        h_t2 = self.t2_encoder(x_t2)
        h_adc = self.adc_encoder(x_adc)

        # concatenate latent representations
        h_cat = torch.cat([h_t1, h_t2, h_adc], dim=1)   # (B, 3 * hidden_dim)

        # gating weights
        gate_logits = self.gate_net(h_cat)              # (B, 3)
        gates = torch.softmax(gate_logits, dim=1)       # each row sums to 1

        # weighted fusion
        h_fused = (
            gates[:, 0:1] * h_t1 +
            gates[:, 1:2] * h_t2 +
            gates[:, 2:3] * h_adc
        )

        # final logit
        logit = self.classifier(h_fused).squeeze(1)     # (B,)

        return logit, gates

In [268]:
def evaluate_model(model, data_loader, device):
    model.eval()

    all_y = []
    all_prob = []
    all_pred = []
    all_gates = []

    with torch.no_grad():
        for x_t1, x_t2, x_adc, y in data_loader:
            x_t1 = x_t1.to(device)
            x_t2 = x_t2.to(device)
            x_adc = x_adc.to(device)
            y = y.to(device)

            logits, gates = model(x_t1, x_t2, x_adc)
            probs = torch.sigmoid(logits)
            preds = (probs >= 0.5).float()

            all_y.extend(y.cpu().numpy())
            all_prob.extend(probs.cpu().numpy())
            all_pred.extend(preds.cpu().numpy())
            all_gates.append(gates.cpu().numpy())

    all_y = np.array(all_y)
    all_prob = np.array(all_prob)
    all_pred = np.array(all_pred)
    all_gates = np.vstack(all_gates)

    acc = accuracy_score(all_y, all_pred)

    # AUC requires both classes present
    if len(np.unique(all_y)) == 2:
        auc = roc_auc_score(all_y, all_prob)
    else:
        auc = np.nan

    mean_gates = all_gates.mean(axis=0)
    var_gates = all_gates.var(axis=0)

    return {
        "acc": acc,
        "auc": auc,
        "mean_gates": mean_gates,
        "var_gates": var_gates,
        "y_true": all_y,
        "y_prob": all_prob,
        "y_pred": all_pred,
        "all_gates": all_gates
    }


def train_model(
    model,
    train_loader,
    test_loader,
    device,
    lr=1e-3,
    weight_decay=1e-4,
    num_epochs=100
):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for x_t1, x_t2, x_adc, y in train_loader:
            x_t1 = x_t1.to(device)
            x_t2 = x_t2.to(device)
            x_adc = x_adc.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            logits, gates = model(x_t1, x_t2, x_adc)
            loss = criterion(logits, y)

            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * y.size(0)

        epoch_loss /= len(train_loader.dataset)

        train_result = evaluate_model(model, train_loader, device)
        test_result = evaluate_model(model, test_loader, device)

        if epoch == 0 or (epoch+1) % 1 == 0:
            print(
            f"Epoch {epoch+1:03d} | "
            f"Loss: {epoch_loss:.4f} | "
            f"Train AUC: {train_result['auc']:.4f} | "
            f"Train ACC: {train_result["acc"]:.4f} | "
            # check confusion matrix if
            # f"Train Confusion Matrix:\n{pd.crosstab(train_result['y_true'], train_result['y_pred'], rownames=['True'], colnames=['Pred'])}\n"
            f"Test AUC: {test_result['auc']:.4f} | "
            f"Test ACC: {test_result['acc']:.4f}"
            # f"Test Confusion Matrix:\n{pd.crosstab(test_result['y_true'], test_result['y_pred'], rownames=['True'], colnames=['Pred'])}\n"
            )
            print(
                f"Mean train gates [T1, T2, ADC]: {np.round(train_result['mean_gates'], 4)}",
                f"Std train gates [T1, T2, ADC]: {np.round(train_result['var_gates'], 4)}"
            )
    return model

In [282]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GatedAdaptiveFusionNet(
    t1_dim=X_t1_train.shape[1],
    t2_dim=X_t2_train.shape[1],
    adc_dim=X_adc_train.shape[1],
    hidden_dim=8,
    gate_hidden_dim=8,
    clf_hidden_dim=8,
    dropout=0.4
).to(device)

model = train_model(
    model,
    train_loader,
    test_loader,
    device,
    lr=1e-3,
    weight_decay=1e-4,
    num_epochs=200
)



Epoch 001 | Loss: 0.7531 | Train AUC: 0.7863 | Train ACC: 0.2357 | Test AUC: 0.7869 | Test ACC: 0.2273
Mean train gates [T1, T2, ADC]: [0.3458 0.3513 0.3029] Std train gates [T1, T2, ADC]: [1.e-04 3.e-04 3.e-04]
Epoch 002 | Loss: 0.7281 | Train AUC: 0.8617 | Train ACC: 0.3840 | Test AUC: 0.8275 | Test ACC: 0.4394
Mean train gates [T1, T2, ADC]: [0.3333 0.3481 0.3186] Std train gates [T1, T2, ADC]: [0.0002 0.0002 0.0004]
Epoch 003 | Loss: 0.6865 | Train AUC: 0.8706 | Train ACC: 0.5171 | Test AUC: 0.8562 | Test ACC: 0.5909
Mean train gates [T1, T2, ADC]: [0.3181 0.3438 0.3382] Std train gates [T1, T2, ADC]: [7.0e-04 1.0e-04 1.1e-03]
Epoch 004 | Loss: 0.6571 | Train AUC: 0.8857 | Train ACC: 0.8745 | Test AUC: 0.8784 | Test ACC: 0.8030
Mean train gates [T1, T2, ADC]: [0.2958 0.3393 0.3649] Std train gates [T1, T2, ADC]: [0.0023 0.0003 0.0042]
Epoch 005 | Loss: 0.5929 | Train AUC: 0.8919 | Train ACC: 0.8821 | Test AUC: 0.8954 | Test ACC: 0.8182
Mean train gates [T1, T2, ADC]: [0.2671 0.3294